**Navigation** : [Index](README.md) | [<< Précédent](10g_Speculative_Decoding.ipynb) | [Suivant >>](11_Quantization.ipynb) | [Version Python](10h_Roofline_Memory_Bandwidth.ipynb)

# 10h — Toit mémoire et bande passante : pourquoi le décodage est limité par la bande passante

Le modèle roofline compare une puissance de calcul $P$ à une bande passante $B$ selon l'intensité en FLOP/octet. Nous mesurons deux noyaux CPU puis projetons une couche linéaire : aucun LLM n'est exécuté. Le régime mémoire ou calcul dépend de la crête mesurée et du modèle choisi.

**Objectifs** : mesurer deux références empiriques, calculer intensité et crête, comparer batchs et projection de vérification spéculative. **Prérequis** : tableaux, multiplication matricielle et [mécanique d'inférence](10b_Inference_Mechanics.ipynb). **Durée estimée** : 45 minutes. **Environnement** : CPU, Float32 ; .NET 9+ et TorchSharp-cpu 0.107.0.

## Les deux plafonds

Le modèle utilise $\min(P, B i)$ où $i$ est l'intensité arithmétique. Les références ci-dessous sont les débits empiriques d'une somme et d'un produit matriciel CPU, pas des pics physiques certifiés. Les tableaux d'entrée et l'échauffement précèdent les chronomètres ; les résultats temporaires sont alloués et libérés dans les opérations mesurées. Les noyaux CPU sont synchrones. Les caches, tailles, bibliothèques et threads influencent les résultats.

In [1]:
#r "nuget:TorchSharp-cpu,0.107.0"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Diagnostics;
using System.Globalization;
using System.Text;
using System.Security;
using TorchSharp;
using static TorchSharp.torch;
using Microsoft.DotNet.Interactive.Formatting;
CultureInfo.DefaultThreadCurrentCulture=CultureInfo.InvariantCulture;
torch.set_num_threads(1);torch.manual_seed(42);
var left=torch.tensor(new float[]{1,2,3,4,5,6}).reshape(2,3);var right=torch.tensor(new float[]{7,8,9,10,11,12}).reshape(3,2);var fixture=torch.matmul(left,right);if(!fixture.data<float>().ToArray().SequenceEqual(new float[]{58,64,139,154})||left.sum().item<float>()!=21)throw new Exception("Oracle natif incorrect");
const int n=32_000_000,m=1024;var a=torch.rand(new long[]{n},dtype:ScalarType.Float32);var A=torch.rand(new long[]{m,m},dtype:ScalarType.Float32);var Bm=torch.rand(new long[]{m,m},dtype:ScalarType.Float32);
using(var warm=a.sum()){} using(var warm=torch.matmul(A,Bm)){}
var sw=Stopwatch.StartNew();float somme=0;for(int j=0;j<10;j++){using var r=a.sum();somme=r.item<float>();}sw.Stop();double bw=n*4d*10/sw.Elapsed.TotalSeconds/1e9;
sw.Restart();float checksum=0;for(int j=0;j<20;j++){using var r=torch.matmul(A,Bm);using var element=r[0,0];checksum=element.item<float>();}sw.Stop();double pf=2d*m*m*m*20/sw.Elapsed.TotalSeconds/1e9;
if(a.element_size()!=4||!double.IsFinite(bw)||!double.IsFinite(pf)||bw<=0||pf<=0)throw new Exception("Mesure invalide");
Console.WriteLine($".NET {Environment.Version}; TorchSharp {typeof(Tensor).Assembly.GetName().Version}; libtorch CPU 2.10.0; Float32 {a.element_size()} octets; threads={torch.get_num_threads()}");Console.WriteLine($"Somme {n} × 10 : {somme:F3}; B={bw:F3} Go/s");Console.WriteLine($"Matmul {m}³ × 20 : C[0,0]={checksum:F3}; P={pf:F3} GFLOP/s; oracle natif 2×3 validé.");

Installed Packages TorchSharp-cpu, 0.107.0

Loading extensions from `/workspace/coursia-toolchains/nuget/skiasharp/2.88.6/interactive-extensions/dotnet/SkiaSharp.DotNet.Interactive.dll`

.NET 10.0.14; TorchSharp 0.107.0.0; libtorch CPU 2.10.0; Float32 4 octets; threads=1


Somme 32000000 × 10 : 15998257.000; B=7.767 Go/s


Matmul 1024³ × 20 : C[0,0]=255.538; P=109.349 GFLOP/s; oracle natif 2×3 validé.



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy



**Lecture du résultat.** Les sommes et produits affichent leur checksum et les débits mesurés. Leurs références varient selon l'environnement ; le quotient $P/B$ détermine la crête de ce modèle, sans mesurer un LLM ni certifier la bande passante matérielle.

## La courbe roofline

Pour une intensité $i$, le plafond modélisé est $\min(P,Bi)$ ; les branches se croisent à $i^*=P/B$. La figure parcourt 200 points de 0,1 à 1000 FLOP/octet en échelle logarithmique.

In [2]:
record RoofSvg(string Content);
Formatter.Register<RoofSvg>((v,w)=>w.Write(v.Content),"text/html");
string F(double v)=>v.ToString("0.###",CultureInfo.InvariantCulture);string E(string v)=>SecurityElement.Escape(v)??"";
string Graph(string titre,string xlab,string ylab,(string nom,string couleur,double[] x,double[] y)[] series,bool logX,bool logY,double? crete=null){
 var allx=series.SelectMany(s=>s.x).Select(v=>logX?Math.Log10(v):v).ToArray();var ally=series.SelectMany(s=>s.y).Select(v=>logY?Math.Log10(v):v).ToArray();double xmin=allx.Min(),xmax=allx.Max(),ymin=ally.Min(),ymax=ally.Max();if(xmax==xmin)xmax=xmin+1;if(ymax==ymin)ymax=ymin+1;
 double X(double v)=>120+550*((logX?Math.Log10(v):v)-xmin)/(xmax-xmin);double Y(double v)=>325-250*((logY?Math.Log10(v):v)-ymin)/(ymax-ymin);
 var s=new StringBuilder($"<svg xmlns='http://www.w3.org/2000/svg' width='760' height='420' viewBox='0 0 760 420'><rect width='100%' height='100%' fill='white'/><text x='380' y='23' text-anchor='middle'>{E(titre)}</text><path d='M120 75 V325 H670' fill='none' stroke='black'/>");
 for(int i=0;i<=4;i++){double x=120+i*137.5,y=325-i*62.5;double xv=xmin+(xmax-xmin)*i/4,yv=ymin+(ymax-ymin)*i/4;if(logX)xv=Math.Pow(10,xv);if(logY)yv=Math.Pow(10,yv);s.Append($"<path d='M120 {F(y)} H670' stroke='#ddd'/><text x='{F(x)}' y='345' text-anchor='middle'>{xv:G3}</text><text x='113' y='{F(y+4)}' text-anchor='end'>{yv:G3}</text>");}
 if(crete is double cr && cr>0 && (logX?Math.Log10(cr):cr)>=xmin && (logX?Math.Log10(cr):cr)<=xmax)s.Append($"<path d='M{F(X(cr))} 75 V325' stroke='#b00020' stroke-dasharray='5,4'/><text x='{F(X(cr)+4)}' y='70' fill='#b00020'>crête</text>");
 for(int j=0;j<series.Length;j++){var z=series[j];s.Append($"<polyline fill='none' stroke='{z.couleur}' stroke-width='2' points='{string.Join(" ",z.x.Select((x,i)=>F(X(x))+","+F(Y(z.y[i]))))}'/>");s.Append($"<text x='{100+j*200}' y='53' fill='{z.couleur}'>{E(z.nom)}</text>");if(z.x.Length<=10)for(int q=0;q<z.x.Length;q++){s.Append($"<circle cx='{F(X(z.x[q]))}' cy='{F(Y(z.y[q]))}' r='3' fill='{z.couleur}'/>");if(z.nom=="lots testés")s.Append($"<text x='{F(X(z.x[q])+5)}' y='{F(Y(z.y[q])-5)}' font-size='10'>{Math.Pow(2,q)}</text>");}}
 s.Append($"<text x='380' y='390' text-anchor='middle'>{E(xlab)}</text><text x='14' y='200' transform='rotate(-90 14 200)' text-anchor='middle'>{E(ylab)}</text></svg>");return s.ToString();}
double iStar=pf/bw;double[] intensites=Enumerable.Range(0,200).Select(k=>Math.Pow(10,-1+4*k/199d)).ToArray();double[] roof=intensites.Select(v=>Math.Min(pf,bw*v)).ToArray();double[] rampe=intensites.Select(v=>bw*v).ToArray();double[] plateau=intensites.Select(v=>pf).ToArray();
Console.WriteLine($"Crête P/B={iStar:F3} FLOP/octet; 200 points de 0,1 à 1000.");display(new RoofSvg(Graph("Roofline empirique", "Intensité (FLOP/octet, log)","Performance (GFLOP/s, log)",new[]{("B × i","#0645ad",intensites,rampe),("P","#a00",intensites,plateau),("min(P, B × i)","#222",intensites,roof)},true,true,iStar)));

Crête P/B=14.079 FLOP/octet; 200 points de 0,1 à 1000.


Roofline empirique 0.1 0.777 1 7.77 10 77.7 100 777 1E+03 7.77E+03 crête <polyline fill='none' stroke='#0645ad' stroke-width='2' points='120,325 122.764,323.744 125.528,322.487 128.291,321.231 131.055,319.975 133.819,318.719 136.583,317.462 139.347,316.206 142.111,314.95 144.874,313.693 147.638,312.437 150.402,311.181 153.166,309.925 155.93,308.668 158.693,307.412 161.457,306.156 164.221,304.899 166.985,303.643 169.749,302.387 172.513,301.131 175.276,299.874 178.04,298.618 180.804,297.362 183.568,296.106 186.332,294.849 189.095,293.593 191.859,292.337 194.623,291.08 197.387,289.824 200.151,288.568 202.915,287.312 205.678,286.055 208.442,284.799 211.206,283.543 213.97,282.286 216.734,281.03 219.497,279.774 222.261,278.518 225.025,277.261 227.789,276.005 230.553,274.749 233.317,273.492 236.08,272.236 238.844,270.98 241.608,269.724 244.372,268.467 247.136,267.211 249.899,265.955 252.663,264.698 255.427,263.442 258.191,262.186 260.955,260.93 263.719,259.673 266.482,258.417 269.246,257.161 272.01,255.905 274.774,254.648 277.538,253.392 280.302,252.136 283.065,250.879 285.829,249.623 288.593,248.367 291.357,247.111 294.121,245.854 296.884,244.598 299.648,243.342 302.412,242.085 305.176,240.829 307.94,239.573 310.704,238.317 313.467,237.06 316.231,235.804 318.995,234.548 321.759,233.291 324.523,232.035 327.286,230.779 330.05,229.523 332.814,228.266 335.578,227.01 338.342,225.754 341.106,224.497 343.869,223.241 346.633,221.985 349.397,220.729 352.161,219.472 354.925,218.216 357.688,216.96 360.452,215.704 363.216,214.447 365.98,213.191 368.744,211.935 371.508,210.678 374.271,209.422 377.035,208.166 379.799,206.91 382.563,205.653 385.327,204.397 388.09,203.141 390.854,201.884 393.618,200.628 396.382,199.372 399.146,198.116 401.91,196.859 404.673,195.603 407.437,194.347 410.201,193.09 412.965,191.834 415.729,190.578 418.492,189.322 421.256,188.065 424.02,186.809 426.784,185.553 429.548,184.296 432.312,183.04 435.075,181.784 437.839,180.528 440.603,179.271 443.367,178.015 446.131,176.759 448.894,175.503 451.658,174.246 454.422,172.99 457.186,171.734 459.95,170.477 462.714,169.221 465.477,167.965 468.241,166.709 471.005,165.452 473.769,164.196 476.533,162.94 479.296,161.683 482.06,160.427 484.824,159.171 487.588,157.915 490.352,156.658 493.116,155.402 495.879,154.146 498.643,152.889 501.407,151.633 504.171,150.377 506.935,149.121 509.698,147.864 512.462,146.608 515.226,145.352 517.99,144.095 520.754,142.839 523.518,141.583 526.281,140.327 529.045,139.07 531.809,137.814 534.573,136.558 537.337,135.302 540.101,134.045 542.864,132.789 545.628,131.533 548.392,130.276 551.156,129.02 553.92,127.764 556.683,126.508 559.447,125.251 562.211,123.995 564.975,122.739 567.739,121.482 570.503,120.226 573.266,118.97 576.03,117.714 578.794,116.457 581.558,115.201 584.322,113.945 587.085,112.688 589.849,111.432 592.613,110.176 595.377,108.92 598.141,107.663 600.905,106.407 603.668,105.151 606.432,103.894 609.196,102.638 611.96,101.382 614.724,100.126 617.487,98.869 620.251,97.613 623.015,96.357 625.779,95.101 628.543,93.844 631.307,92.588 634.07,91.332 636.834,90.075 639.598,88.819 642.362,87.563 645.126,86.307 647.889,85.05 650.653,83.794 653.417,82.538 656.181,81.281 658.945,80.025 661.709,78.769 664.472,77.513 667.236,76.256 670,75'/> B × i <polyline fill='none' stroke='#a00' stroke-width='2' points='120,190.714 122.764,190.714 125.528,190.714 128.291,190.714 131.055,190.714 133.819,190.714 136.583,190.714 139.347,190.714 142.111,190.714 144.874,190.714 147.638,190.714 150.402,190.714 153.166,190.714 155.93,190.714 158.693,190.714 161.457,190.714 164.221,190.714 166.985,190.714 169.749,190.714 172.513,190.714 175.276,190.714 178.04,190.714 180.804,190.714 183.568,190.714 186.332,190.714 189.095,190.714 191.859,190.714 194.623,190.714 197.387,190.714 200.151,190.714 202.915,190.714 205.678,190.714 208.442,190.714 211.206,190.714 213.97,190.714 216.734,190.714 219.497,190.714 222.261,190.714 225.025,190.714 227.789,190.714 230.553,190.714 233.317,190.714 

**Lecture du résultat.** La rampe $Bi$ et le plateau $P$ se rencontrent à la crête affichée. La courbe est une projection idéale, non une mesure de performance pour chaque intensité.

## Où se place une couche du transformer ?

Le modèle utilise $W=4096^2$ poids et $A=4096$ activations, en Float32 : $2Wb$ FLOP et $4W+4Ab$ octets. Pour $b=1$ et $b=512$, comparons l'intensité à la crête mesurée ; les régimes dépendent de $P/B$.

In [3]:
const double W=4096d*4096,Ae=4096;
double Intensite(double batch)=>2*W*batch/(4*W+4*Ae*batch);
double iGemv=Intensite(1),iPrefill=Intensite(512);foreach(var (nom,v) in new[]{("décodage b=1",iGemv),("prefill b=512",iPrefill)})Console.WriteLine($"{nom} : {v:F6} FLOP/octet, régime projeté : {(v>=iStar?"calcul":"mémoire")}");Console.WriteLine($"Crête mesurée : {iStar:F3} FLOP/octet");

décodage b=1 : 0.499878 FLOP/octet, régime projeté : mémoire


prefill b=512 : 227.555556 FLOP/octet, régime projeté : calcul


Crête mesurée : 14.079 FLOP/octet


**Lecture du résultat.** Les intensités structurelles sont environ 0,499878 pour $b=1$ et 227,556 FLOP/octet pour $b=512$. Le programme annonce le régime projeté selon la crête observée : aucun classement n'est universel.

### Exercice 1 — Intensité arithmétique d'un batch

Écrivez une fonction de $batch$, $W$ et $act$ : $2Wb/(4W+4Ab)$. Pour les fixtures, $W>0$, $b\ge 0$, $act\ge 0$ ; une réponse doit être finie. Le batch nul donne zéro.

In [4]:
Func<double,double,double,double?> intensiteEx=null; // TODO étudiant : calculer FLOPs / octets
if(intensiteEx is null)Console.WriteLine("Exercice 1 à compléter : intensité du lot.");else {foreach(var (b,w,a,attendu) in new[]{(1d,4d,0d,.5),(2d,4d,2d,.5),(0d,4d,3d,0d),(1d,4d,4d,.25)}){double? x=intensiteEx(b,w,a);if(x is null||!double.IsFinite(x.Value)||Math.Abs(x.Value-attendu)>1e-10)throw new Exception("Intensité incorrecte");}Console.WriteLine("Quatre intensités vérifiées, dont batch nul.");}

Exercice 1 à compléter : intensité du lot.


## Le levier : la taille du batch

Amortir les poids sur plusieurs tokens augmente l'intensité dans ce modèle. Les activations demeurent dans le dénominateur : la croissance n'est donc pas exactement proportionnelle au batch. La figure et la table projettent neuf tailles ; elles ne mesurent pas une latence de service.

In [5]:
int[] batches={1,2,4,8,16,32,64,128,256};double[] ib=batches.Select(b=>Intensite(b)).ToArray();double[] pb=ib.Select(v=>Math.Min(pf,bw*v)).ToArray();int? seuil=batches.Where((b,j)=>ib[j]>=iStar).Select(b=>(int?)b).FirstOrDefault();
Console.WriteLine($"Premier batch testé atteignant la crête : {seuil?.ToString()??"aucun"}");for(int j=0;j<batches.Length;j++)Console.WriteLine($"b={batches[j],3}: intensité={ib[j],8:F3}, projection={pb[j],8:F3} GFLOP/s");display(new RoofSvg(Graph("Projection des lots", "Intensité (FLOP/octet, log)","Performance (GFLOP/s, log)",new[]{("roofline","#222",intensites,roof),("lots testés","#087b30",ib,pb)},true,true)));

Premier batch testé atteignant la crête : 32


b=  1: intensité=   0.500, projection=   3.882 GFLOP/s


b=  2: intensité=   1.000, projection=   7.763 GFLOP/s


b=  4: intensité=   1.998, projection=  15.518 GFLOP/s


b=  8: intensité=   3.992, projection=  31.007 GFLOP/s


b= 16: intensité=   7.969, projection=  61.892 GFLOP/s


b= 32: intensité=  15.876, projection= 109.349 GFLOP/s


b= 64: intensité=  31.508, projection= 109.349 GFLOP/s


b=128: intensité=  62.061, projection= 109.349 GFLOP/s


b=256: intensité= 120.471, projection= 109.349 GFLOP/s


Projection des lots 0.1 0.777 1 2.68 10 9.22 100 31.7 1E+03 109 <polyline fill='none' stroke='#222' stroke-width='2' points='120,325 122.764,322.661 125.528,320.322 128.291,317.984 131.055,315.645 133.819,313.306 136.583,310.967 139.347,308.628 142.111,306.289 144.874,303.951 147.638,301.612 150.402,299.273 153.166,296.934 155.93,294.595 158.693,292.257 161.457,289.918 164.221,287.579 166.985,285.24 169.749,282.901 172.513,280.562 175.276,278.224 178.04,275.885 180.804,273.546 183.568,271.207 186.332,268.868 189.095,266.53 191.859,264.191 194.623,261.852 197.387,259.513 200.151,257.174 202.915,254.835 205.678,252.497 208.442,250.158 211.206,247.819 213.97,245.48 216.734,243.141 219.497,240.803 222.261,238.464 225.025,236.125 227.789,233.786 230.553,231.447 233.317,229.108 236.08,226.77 238.844,224.431 241.608,222.092 244.372,219.753 247.136,217.414 249.899,215.076 252.663,212.737 255.427,210.398 258.191,208.059 260.955,205.72 263.719,203.381 266.482,201.043 269.246,198.704 272.01,196.365 274.774,194.026 277.538,191.687 280.302,189.349 283.065,187.01 285.829,184.671 288.593,182.332 291.357,179.993 294.121,177.654 296.884,175.316 299.648,172.977 302.412,170.638 305.176,168.299 307.94,165.96 310.704,163.622 313.467,161.283 316.231,158.944 318.995,156.605 321.759,154.266 324.523,151.927 327.286,149.589 330.05,147.25 332.814,144.911 335.578,142.572 338.342,140.233 341.106,137.895 343.869,135.556 346.633,133.217 349.397,130.878 352.161,128.539 354.925,126.2 357.688,123.862 360.452,121.523 363.216,119.184 365.98,116.845 368.744,114.506 371.508,112.168 374.271,109.829 377.035,107.49 379.799,105.151 382.563,102.812 385.327,100.473 388.09,98.135 390.854,95.796 393.618,93.457 396.382,91.118 399.146,88.779 401.91,86.44 404.673,84.102 407.437,81.763 410.201,79.424 412.965,77.085 415.729,75 418.492,75 421.256,75 424.02,75 426.784,75 429.548,75 432.312,75 435.075,75 437.839,75 440.603,75 443.367,75 446.131,75 448.894,75 451.658,75 454.422,75 457.186,75 459.95,75 462.714,75 465.477,75 468.241,75 471.005,75 473.769,75 476.533,75 479.296,75 482.06,75 484.824,75 487.588,75 490.352,75 493.116,75 495.879,75 498.643,75 501.407,75 504.171,75 506.935,75 509.698,75 512.462,75 515.226,75 517.99,75 520.754,75 523.518,75 526.281,75 529.045,75 531.809,75 534.573,75 537.337,75 540.101,75 542.864,75 545.628,75 548.392,75 551.156,75 553.92,75 556.683,75 559.447,75 562.211,75 564.975,75 567.739,75 570.503,75 573.266,75 576.03,75 578.794,75 581.558,75 584.322,75 587.085,75 589.849,75 592.613,75 595.377,75 598.141,75 600.905,75 603.668,75 606.432,75 609.196,75 611.96,75 614.724,75 617.487,75 620.251,75 623.015,75 625.779,75 628.543,75 631.307,75 634.07,75 636.834,75 639.598,75 642.362,75 645.126,75 647.889,75 650.653,75 653.417,75 656.181,75 658.945,75 661.709,75 664.472,75 667.236,75 670,75'/> roofline lots testés 1 2 4 8 16 32 64 128 256 Intensité (FLOP/octet, log) Performance (GFLOP/s, log)

**Lecture du résultat.** Le seuil affiché est le premier batch de la grille testée à atteindre la crête, ou aucun ; il ne prétend pas être le plus petit entier possible. Plus de débit projeté ne garantit pas une meilleure latence individuelle.

### Exercice 2 — La courbe roofline

Calculez $\min(P,Bi)$ pour $i,P\ge0$ et $B>0$. Les fixtures testent les deux branches et l'intensité nulle.

In [6]:
Func<double,double,double,double?> rooflineEx=null; // TODO étudiant : minimum des branches
if(rooflineEx is null)Console.WriteLine("Exercice 2 à compléter : minimum des deux plafonds.");else {foreach(var (i,p,b,attendu) in new[]{(1d,10d,2d,2d),(10d,10d,2d,10d),(0d,10d,2d,0d)}){double? x=rooflineEx(i,p,b);if(x is null||!double.IsFinite(x.Value)||Math.Abs(x.Value-attendu)>1e-10)throw new Exception("Roofline incorrect");}Console.WriteLine("Deux branches et intensité nulle vérifiées.");}

Exercice 2 à compléter : minimum des deux plafonds.


## Décodage spéculatif : une projection de vérification

Pour comparer $k=1$ à $8$, on approxime l'intensité de vérification par $k$ fois celle de GEMV. Cette projection n'exécute pas huit forwards de LLM et néglige le coût du draft, les rejets, le KV-cache, les activations et les chevauchements. Elle éclaire une composante possible du débit, pas un gain garanti de latence.

In [7]:
double[] ks=Enumerable.Range(1,8).Select(k=>(double)k).ToArray();double[] iv=ks.Select(k=>iGemv*k).ToArray();double[] pv=iv.Select(v=>Math.Min(pf,bw*v)).ToArray();
for(int j=0;j<ks.Length;j++)Console.WriteLine($"k={ks[j]} : intensité approximative={iv[j]:F3}, projection={pv[j]:F3} GFLOP/s ({(iv[j]>=iStar?"calcul":"mémoire")})");display(new RoofSvg(Graph("Vérification spéculative projetée", "k (tokens vérifiés)","Performance projetée (GFLOP/s)",new[]{("projection","#0645ad",ks,pv),("P","#a00",ks,ks.Select(_=>pf).ToArray())},false,false)));

k=1 : intensité approximative=0.500, projection=3.882 GFLOP/s (mémoire)


k=2 : intensité approximative=1.000, projection=7.765 GFLOP/s (mémoire)


k=3 : intensité approximative=1.500, projection=11.647 GFLOP/s (mémoire)


k=4 : intensité approximative=2.000, projection=15.530 GFLOP/s (mémoire)


k=5 : intensité approximative=2.499, projection=19.412 GFLOP/s (mémoire)


k=6 : intensité approximative=2.999, projection=23.295 GFLOP/s (mémoire)


k=7 : intensité approximative=3.499, projection=27.177 GFLOP/s (mémoire)


k=8 : intensité approximative=3.999, projection=31.060 GFLOP/s (mémoire)


Vérification spéculative projetée 1 3.88 2.75 30.2 4.5 56.6 6.25 83 8 109 projection P k (tokens vérifiés) Performance projetée (GFLOP/s)

## Lecture du résultat

La table donne les huit débits **projetés** et leur régime selon la crête. La réutilisation des poids peut élever l'intensité de vérification ; le coût réel dépend aussi du draft et de l'acceptation des tokens. Ni le batching ni la spéculation ne garantit un gain dans toutes les configurations.

## Limites

Les références CPU dépendent des caches, tailles, threads et bibliothèques. La couche modélisée ne comprend ni tout le trafic d'un modèle ni le KV-cache ; son trafic d'activations est inclus dans $4W+4Ab$, mais celui de la vérification spéculative est simplifié. Le classement mémoire/calcul doit être lu en fonction de $P/B$ et ne se transpose pas automatiquement à un GPU.

### Exercice 3 — Le point de crête

Calculez $P/B$ pour $P\ge0$ et $B>0$ ; une puissance nulle donne une crête nulle. Les fixtures refusent une réponse constante ou non finie.

In [8]:
Func<double,double,double?> creteEx=null; // TODO étudiant : P/B
if(creteEx is null)Console.WriteLine("Exercice 3 à compléter : point de crête.");else {foreach(var (p,b,attendu) in new[]{(10d,2d,5d),(6d,3d,2d),(0d,2d,0d)}){double? x=creteEx(p,b);if(x is null||!double.IsFinite(x.Value)||Math.Abs(x.Value-attendu)>1e-10)throw new Exception("Crête incorrecte");}Console.WriteLine("Trois crêtes vérifiées, dont puissance nulle.");}

Exercice 3 à compléter : point de crête.


## Exercices

Les trois exercices répartis portent sur [la roofline](#exercice-2--la-courbe-roofline), [l’intensité](#exercice-1--intensité-arithmétique-dun-batch) et [la crête](#exercice-3--le-point-de-crête). Les stubs restent à compléter.

## À retenir

- La crête $P/B$ compare deux références empiriques, dépendantes de l'environnement.
- Une couche de $4096^2$ poids a dans ce modèle une intensité de 0,499878 à $b=1$ et 227,556 à $b=512$.
- Le batch et la vérification spéculative sont des projections ; leurs gains réels requièrent une mesure de bout en bout.

## Vérifiez votre compréhension

1. Pourquoi la crête varie-t-elle d'une machine à l'autre ?
2. Quel coût manque à la projection spéculative ?

<details><summary>Réponses</summary>

1. $P$ et $B$ dépendent des noyaux, caches, threads et matériels utilisés.
2. Le draft, les rejets, le KV-cache et les autres trafics ne sont pas modélisés.

</details>

## Pour aller plus loin

- [10b — Mécanique d'inférence](10b_Inference_Mechanics.ipynb) : KV-cache.
- [10g — Décodage spéculatif](10g_Speculative_Decoding.ipynb) : mécanisme d'acceptation.
- [11 — Quantification](11_Quantization.ipynb) : précision et coût mémoire.